# Notebook 04 — Baseline Experiments

Runs all 5 baselines (B0–B4) on DTAH-Bench-50 (pilot) and saves scores.

**GPU required** for B4 (IFC-GraphRAG-DT) which runs SDXL + TripoSR.
B0–B3 run on CPU.

This notebook produces the main result table for Paper 2.

In [ ]:
import os, sys, json, time
REPO = 'ifc-graphrag-dt'
if os.path.exists(REPO): !cd {REPO} && git pull --quiet
else: !git clone https://github.com/aiwithprashant/ifc-graphrag-dt.git
sys.path.insert(0, REPO)

# Mount Google Drive for output persistence
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_OUT = '/content/drive/MyDrive/ifc_graphrag_dt_outputs'
    os.makedirs(DRIVE_OUT, exist_ok=True)
    print(f'Drive mounted. Outputs → {DRIVE_OUT}')
except ImportError:
    DRIVE_OUT = f'{REPO}/outputs'
    print('Not in Colab — saving locally.')

os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'  # replace or use Colab Secrets

from pathlib import Path
from benchmark.dtah_bench import DTAHBench
from pipeline.layer1_retriever.ifc_graph_builder import IFCGraphBuilder
from pipeline.layer1_retriever.graph_embedder import GraphEmbedder
from pipeline.layer1_retriever.khop_traversal import KHopTraversal
from pipeline.layer2_spec_gen.scene_spec_generator import SceneSpecGenerator
from pipeline.baselines.b0_no_grounding import B0NoGrounding
from pipeline.baselines.b1_llm_only import B1LLMOnly
from pipeline.baselines.b2_flat_rag import B2FlatRAG
from pipeline.baselines.b3_ifc_lookup import B3IFCLookup
from evaluation.metrics.kcs_dt import KCSDTScorer
from evaluation.metrics.ggs import GGSScorer
from evaluation.stage_a.spec_evaluator import StageAEvaluator
import yaml

print('All imports OK')

In [ ]:
# ── Cell 2: Load graph and embedder ──────────────────────────────────────────
IFC_PATH    = f'{REPO}/benchmark/ifc_reference_models/duplex.ifc'
GRAPH_CACHE = f'{REPO}/outputs/graphs/ifc_graph.json'
EMB_CACHE   = f'{REPO}/outputs/embedders/graph_embedder'

if Path(GRAPH_CACHE).exists():
    G = IFCGraphBuilder.load_graph(GRAPH_CACHE)
else:
    builder = IFCGraphBuilder(IFC_PATH)
    G = builder.build()
    builder.save_graph(GRAPH_CACHE)

if Path(EMB_CACHE).exists():
    embedder = GraphEmbedder.load(EMB_CACHE)
else:
    embedder = GraphEmbedder()
    embedder.fit(G)
    embedder.save(EMB_CACHE)

with open(f'{REPO}/pipeline/configs/ifc_config.yaml') as f:
    ifc_config = yaml.safe_load(f)

print(f'Graph: {G.number_of_nodes()} nodes | Embedder: {len(embedder._node_ids)} nodes')

In [ ]:
# ── Cell 3: Initialise all baselines and evaluation ──────────────────────────
TIER_DEPTHS = {1: 1, 2: 2, 3: 4}  # k-hop depth per tier

llm_config = {'llm_provider': 'anthropic', 'model': 'claude-sonnet-4-20250514',
              'max_tokens': 2048, 'temperature': 0.0,
              'output_dir': f'{REPO}/outputs/specs'}

b0 = B0NoGrounding({'output_dir': f'{REPO}/outputs/meshes'})
b1 = B1LLMOnly(llm_config)
b2 = B2FlatRAG(llm_config, graph_embedder=embedder)
b3 = B3IFCLookup(llm_config)
b4_spec_gen = SceneSpecGenerator(llm_config)  # B4 uses full pipeline

kcs_scorer  = KCSDTScorer()
ggs_scorer  = GGSScorer()
stage_a_eval = StageAEvaluator()

bench = DTAHBench(pilot_mode=True)
all_prompts = bench.load_all()
print(f'Running on {len(all_prompts)} pilot prompts (DTAH-Bench-50)')

In [ ]:
# ── Cell 4: Run B0, B1, B2, B3 (CPU baselines) ───────────────────────────────
# These produce scene specs only (no 3D mesh) — scored at Stage A

baseline_results = {b: [] for b in ['b0','b1','b2','b3']}

for p in all_prompts:
    pid, prompt, tier = p['id'], p['prompt'], p['tier']

    # GraphRAG subgraph (used as proxy GT for now)
    seeds     = embedder.retrieve_seeds(prompt, top_k=5)
    seed_ids  = [s['node_id'] for s in seeds]
    depth     = TIER_DEPTHS[tier]
    trav      = KHopTraversal(G, max_depth=depth, bidirectional=True)
    graph_sg  = trav.traverse(seed_ids).subgraph

    # B0
    spec_b0 = b0.run(prompt, pid)
    baseline_results['b0'].append({'prompt_id': pid, 'tier': tier,
        'entities': len(spec_b0.get('entities',[])),
        'relations': len(spec_b0.get('relations',[])),
        'baseline': 'B0', 'status': 'ok'})

    # B1 (LLM only)
    try:
        spec_b1 = b1.run(prompt, pid)
        baseline_results['b1'].append({'prompt_id': pid, 'tier': tier,
            'entities': len(spec_b1.get('entities',[])),
            'relations': len(spec_b1.get('relations',[])),
            'baseline': 'B1', 'status': 'ok'})
    except Exception as e:
        baseline_results['b1'].append({'prompt_id': pid, 'tier': tier, 'baseline': 'B1', 'status': str(e)})

    # B2 (Flat RAG)
    spec_b2 = b2.run(prompt, G, pid)
    baseline_results['b2'].append({'prompt_id': pid, 'tier': tier,
        'entities': len(spec_b2.get('entities',[])),
        'relations': len(spec_b2.get('relations',[])),
        'baseline': 'B2', 'status': 'ok'})

    # B3 (IFC Lookup)
    spec_b3 = b3.run(prompt, G, ifc_config, pid)
    baseline_results['b3'].append({'prompt_id': pid, 'tier': tier,
        'entities': len(spec_b3.get('entities',[])),
        'relations': len(spec_b3.get('relations',[])),
        'baseline': 'B3', 'status': 'ok'})

    print(f'[{pid}] B0:{len(spec_b0.get("entities",[]))}e | B2:{len(spec_b2.get("entities",[]))}e '
          f'B3:{len(spec_b3.get("entities",[]))}e', end='  \r')

print(f'\n✓ CPU baselines complete.')

In [ ]:
# ── Cell 5: Run B4 — IFC-GraphRAG-DT full pipeline (GPU) ────────────────────
# Checks for GPU, uses dry_run=True if not available
import torch
HAS_GPU = torch.cuda.is_available()
print(f'GPU available: {HAS_GPU}')
if HAS_GPU: print(f'GPU: {torch.cuda.get_device_name(0)}')

if HAS_GPU:
    from pipeline.run_pipeline import IFCGraphRAGDT
    pipeline = IFCGraphRAGDT(config_path=f'{REPO}/pipeline/configs/pipeline_config.yaml',
                              dry_run=False)
else:
    print('No GPU — running B4 in dry_run mode (spec only, no 3D generation)')
    from pipeline.run_pipeline import IFCGraphRAGDT
    pipeline = IFCGraphRAGDT(config_path=f'{REPO}/pipeline/configs/pipeline_config.yaml',
                              dry_run=True)

b4_results = []
for p in all_prompts:
    pid, prompt, tier = p['id'], p['prompt'], p['tier']
    t0 = time.time()
    result = pipeline.run(prompt=prompt, prompt_id=pid, tier=tier)
    elapsed = time.time() - t0
    b4_results.append({'prompt_id': pid, 'tier': tier, 'baseline': 'B4',
        'entities': len(result.spec.get('entities',[])),
        'relations': len(result.spec.get('relations',[])),
        'mesh_path': str(result.mesh_path) if result.mesh_path else None,
        'elapsed_s': round(elapsed, 2), 'status': 'ok'})
    print(f'[{pid}] {elapsed:.1f}s | {len(result.spec.get("entities",[]))} entities', end='  \r')

print(f'\n✓ B4 complete: {len(b4_results)} results')

In [ ]:
# ── Cell 6: Save all baseline results ────────────────────────────────────────
scores_dir = f'{REPO}/outputs/scores'
os.makedirs(scores_dir, exist_ok=True)

for b_name, b_data in baseline_results.items():
    out = f'{scores_dir}/{b_name}_scores.json'
    with open(out, 'w') as f:
        json.dump(b_data, f, indent=2)
    print(f'Saved {out} ({len(b_data)} records)')

out_b4 = f'{scores_dir}/b4_scores.json'
with open(out_b4, 'w') as f:
    json.dump(b4_results, f, indent=2)
print(f'Saved {out_b4} ({len(b4_results)} records)')

# Copy to Drive
if DRIVE_OUT != f'{REPO}/outputs':
    import shutil
    shutil.copytree(scores_dir, f'{DRIVE_OUT}/scores', dirs_exist_ok=True)
    print(f'Scores copied to Google Drive.')

In [ ]:
# ── Cell 7: Entity and relation count summary ─────────────────────────────────
import numpy as np

print('ENTITY COUNTS BY BASELINE AND TIER')
print(f'{"Baseline":<8} {"Tier 1":>10} {"Tier 2":>10} {"Tier 3":>10}  (mean entities per prompt)')
print('-' * 50)

all_baseline_data = {**{b: v for b, v in baseline_results.items()}, 'b4': b4_results}
for b_name, b_data in all_baseline_data.items():
    means = []
    for tier in [1, 2, 3]:
        t_data = [r for r in b_data if r.get('tier') == tier and r.get('status') == 'ok']
        mean_e = np.mean([r.get('entities', 0) for r in t_data]) if t_data else 0
        means.append(mean_e)
    print(f'{b_name.upper():<8} {means[0]:>10.1f} {means[1]:>10.1f} {means[2]:>10.1f}')

print('\nNote: B0 always 0 entities (no grounding). B4 should have most.')